# AZURE SETUP

In [8]:
import os
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient

# 1. Fix the syntax: Use parentheses () for os.environ.get
endpoint = os.environ["AZURE_LANGUAGE_ENDPOINT"]
key = os.environ["AZURE_LANGUAGE_KEY"]

if not endpoint or not key:
    raise ValueError("Please set AZURE_LANGUAGE_ENDPOINT and AZURE_LANGUAGE_KEY in your environment variables.")

# 2. Create the client
credential = AzureKeyCredential(key)
client = TextAnalyticsClient(endpoint=endpoint, credential=credential)

print("Client created successfully!")

Client created successfully!


In [11]:
import os
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient

key = os.environ.get("AZURE_LANGUAGE_KEY")
endpoint = os.environ.get("AZURE_LANGUAGE_ENDPOINT")

In [12]:
client = TextAnalyticsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

documents = [
    "The service was excellent",
    "The food was terrible",
    "The waiter was very friendly",
]

results = client.analyze_sentiment(documents)
for text, result in zip(documents, results):
    print(text, "", result.sentiment)

The service was excellent  positive
The food was terrible  negative
The waiter was very friendly  positive


# Azure Immersive Reader

In [3]:
import os
import requests
from flask import Flask, jsonify, render_template_string
app = Flask(__name__)

In [1]:
import os
import requests

from flask import (
    Flask,
    jsonify,
    render_template_string,
    request
)

from dotenv import load_dotenv

# Load environment variables
load_dotenv()

app = Flask(__name__)

# Azure credentials
TENANT_ID = os.environ["TENANT_ID"]
CLIENT_ID = os.environ["CLIENT_ID"]
CLIENT_SECRET = os.environ["CLIENT_SECRET"]
SUBDOMAIN = os.environ["SUBDOMAIN"]


# HTML frontend
HTML_PAGE = """
<!DOCTYPE html>
<html lang="en">

<head>

    <meta charset="UTF-8">

    <meta name="viewport"
          content="width=device-width, initial-scale=1">

    <title>Azure Immersive Reader</title>

    <script src="https://cdn.jsdelivr.net/npm/@microsoft/immersive-reader-sdk@latest/lib/immersive-reader.umd.min.js">
    </script>

    <style>

        body {
            font-family: Arial;
            margin: 40px;
            background: #f5f5f5;
        }

        .container {
            background: white;
            padding: 30px;
            border-radius: 10px;
            max-width: 800px;
            margin: auto;
        }

        textarea {
            width: 100%;
            height: 180px;
            padding: 10px;
            font-size: 16px;
            box-sizing: border-box;
        }

        button {
            padding: 12px 20px;
            background: #0078d4;
            color: white;
            border: none;
            cursor: pointer;
            margin-top: 15px;
            border-radius: 5px;
        }

        button:hover {
            background: #005a9e;
        }

        #status {
            margin-top: 15px;
        }

    </style>

</head>

<body>

<div class="container">

    <h1>Azure Immersive Reader</h1>

    <p>
        Enter some text below and launch
        Microsoft Immersive Reader.
    </p>

    <textarea id="reading-content">Machine learning enables computers to learn patterns from data and make predictions.

Artificial intelligence is transforming industries by automating tasks and improving decision-making.

Python is widely used for machine learning and artificial intelligence development.</textarea>

    <br>

    <button onclick="launchReader()">
        Open Immersive Reader
    </button>

    <p id="status"></p>

</div>


<script>

async function launchReader() {

    const status = document.getElementById("status");

    status.innerText = "Obtaining Azure access token...";

    try {

        // Get authentication token from Flask backend

        const response = await fetch("/token");

        if (!response.ok) {

            throw new Error("Authentication failed");

        }

        const auth = await response.json();

        // Get text entered by user

        const readingText = document.getElementById(
            "reading-content"
        ).value;

        // Prepare Immersive Reader content

        const content = {

            title: "My Reading Content",

            chunks: [

                {

                    content: readingText,

                    lang: "en"

                }

            ]

        };

        // Launch Immersive Reader

        status.innerText = "Launching Immersive Reader...";

        await ImmersiveReader.launchAsync(

            auth.token,

            auth.subdomain,

            content

        );

        status.innerText = "Immersive Reader closed.";

    }

    catch (error) {

        status.innerText = "Error: " + error.message;

        console.error(error);

    }

}

</script>

</body>

</html>
"""


# Home route
@app.route("/")
def index():

    return render_template_string(HTML_PAGE)


# Azure authentication route
@app.route("/token")
def get_token():

    token_url = (
        f"https://login.microsoftonline.com/"
        f"{TENANT_ID}/oauth2/v2.0/token"
    )

    payload = {

        "client_id": CLIENT_ID,

        "client_secret": CLIENT_SECRET,

        "scope": "https://cognitiveservices.azure.com/.default",

        "grant_type": "client_credentials"

    }

    try:

        response = requests.post(

            token_url,

            data=payload,

            timeout=20

        )

        response.raise_for_status()

        token_data = response.json()

        return jsonify({

            "token": token_data["access_token"],

            "subdomain": SUBDOMAIN

        })

    except requests.RequestException:

        app.logger.exception("Azure token request failed")

        return jsonify({

            "error": "Unable to authenticate with Azure"

        }), 502


# Run application
if __name__ == "__main__":

    app.run(

        host="127.0.0.1",

        port=5000,

        debug=False

    )

KeyError: 'TENANT_ID'

# Python implementation: Analyze an image

In [ ]:
import os
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential

# Azure credentials
ENDPOINT = os.environ.get("AZURE_VISION_ENDPOINT")
KEY = os.environ.get("AZURE_VISION_KEY")

# Create Azure Vision client
client = ImageAnalysisClient(
    endpoint=ENDPOINT,
    credential=AzureKeyCredential(KEY)
)

# Image URL
image_url = "https://aka.ms/azsdk/image-analysis/sample.jpg"

# Analyze image
result = client.analyze_from_url(
    image_url=image_url,
    visual_features=[
        VisualFeatures.CAPTION,
        VisualFeatures.TAGS,
        VisualFeatures.OBJECTS
    ]
)

# Print image description
if result.caption:
    print("Image Description:")
    print(result.caption.text)

# Print tags
if result.tags:
    print("\nDetected Tags:")
    for tag in result.tags.list:
        print(tag.name, tag.confidence)

# Print detected objects
if result.objects:
    print("\nDetected Objects:")
    for obj in result.objects.list:
        print("Object:", obj.tags[0].name)
        print("Confidence:", obj.tags[0].confidence)
        print("Bounding Box:", obj.bounding_box)

ModuleNotFoundError: No module named 'azure'

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get credentials
ENDPOINT = os.environ.get("AZURE_VISION_ENDPOINT")
KEY = os.environ.get("AZURE_VISION_KEY")

if not ENDPOINT or not KEY:
    raise ValueError("Please set AZURE_VISION_ENDPOINT and AZURE_VISION_KEY")

from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential

# Create Azure Vision client
client = ImageAnalysisClient(
    endpoint=ENDPOINT,
    credential=AzureKeyCredential(KEY)
)

# Image URL
image_url = "https://aka.ms/azsdk/image-analysis/sample.jpg"

# Analyze image
result = client.analyze_from_url(
    image_url=image_url,
    visual_features=[
        VisualFeatures.CAPTION,
        VisualFeatures.TAGS,
        VisualFeatures.OBJECTS
    ]
)

# Print image description
if result.caption:
    print("Image Description:")
    print(result.caption.text)

# Print tags
if result.tags:
    print("\nDetected Tags:")
    for tag in result.tags.list:
        print(tag.name, tag.confidence)

# Print detected objects
if result.objects:
    print("\nDetected Objects:")
    for obj in result.objects.list:
        print("Object:", obj.tags[0].name)
        print("Confidence:", obj.tags[0].confidence)
        print("Bounding Box:", obj.bounding_box)

ModuleNotFoundError: No module named 'azure.ai.vision'